# AquaShield AI — Chennai rainfall EDA

This notebook analyzes the actual CSV produced by `scripts/ingest_rainfall.py`. It does not contain pre-filled observations. Run ingestion first, then restart and run all cells. The daily values are ERA5 reanalysis estimates, not direct station readings or flood labels.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path('../data/processed/chennai_daily_rainfall.csv')
if not data_path.exists():
    raise FileNotFoundError('Run: python scripts/ingest_rainfall.py --start 2020-01-01 --end 2025-12-31')
rain = pd.read_csv(data_path, parse_dates=['date'])
rain.head()

In [ ]:
# Quality checks and descriptive profile
quality = {
    'rows': len(rain),
    'date_min': rain['date'].min(),
    'date_max': rain['date'].max(),
    'duplicate_dates': int(rain['date'].duplicated().sum()),
    'missing_precipitation': int(rain['precipitation_mm'].isna().sum()),
    'negative_precipitation': int((rain['precipitation_mm'].dropna() < 0).sum()),
}
quality, rain['precipitation_mm'].describe()

In [ ]:
# Monthly aggregation from observed source rows; not a flood-risk score
monthly = (rain.dropna(subset=['precipitation_mm'])
           .assign(month=lambda df: df['date'].dt.to_period('M').dt.to_timestamp())
           .groupby('month', as_index=False)['precipitation_mm'].sum())
ax = monthly.plot(x='month', y='precipitation_mm', figsize=(12, 4), legend=False, color='#147d68')
ax.set(title='Chennai estimated monthly precipitation', xlabel='Month', ylabel='Precipitation (mm)')
plt.tight_layout()
plt.show()

## Interpretation boundary
Rainfall patterns can describe weather variability at this coordinate. They do not establish where or when waterlogging occurred, whether a road was accessible, or whether floodwater was contaminated. Predictive modeling needs verified event labels and aligned spatial/temporal records, which are not currently configured.